# 02. Download the history
Hourly prices from REN (2019 to June 2026) and 15-minute production by source from REN (2019 to today).

In [1]:
# Import the libraries
import time
import requests
import pandas as pd

In [2]:
# Download one day of REN prices: one row per hour, one column per country
def read_ren_prices_day(day):
    url = "https://servicebus.ren.pt/datahubapi/electricity/ElectricityMarketPricesDaily"
    response = requests.get(url, params={"culture": "en-US", "date": day})
    if response.status_code != 200:
        return pd.DataFrame()
    data = response.json()
    table = pd.DataFrame({"hour": data["xAxis"]["categories"]})
    for serie in data["series"]:
        table["price_" + serie["name"].lower()] = serie["data"]
    table["date"] = day
    return table

# Download one day of REN production: one row per 15 minutes, one column per source
def read_ren_production_day(day):
    url = "https://servicebus.ren.pt/datahubapi/electricity/ElectricityProductionBreakdownDaily"
    response = requests.get(url, params={"culture": "en-US", "date": day})
    if response.status_code != 200:
        return pd.DataFrame()
    data = response.json()
    table = pd.DataFrame({"time": data["xAxis"]["categories"]})
    for serie in data["series"]:
        table[serie["name"]] = serie["data"]
    table["date"] = day
    return table

In [3]:
# Test both functions on one week before downloading everything
test_prices = []
test_production = []
for day in pd.date_range("2025-03-24", "2025-03-31"):
    day_text = day.strftime("%Y-%m-%d")
    test_prices.append(read_ren_prices_day(day_text))
    test_production.append(read_ren_production_day(day_text))

test_prices = pd.concat(test_prices)
test_production = pd.concat(test_production)
print(len(test_prices), "price rows (8 days x 24 hours = 192, minus 1 on the clock change day)")
print(len(test_production), "production rows (8 days x 96 = 768, minus 4 on the clock change day)")
test_production.head()

191 price rows (8 days x 24 hours = 192, minus 1 on the clock change day)
764 production rows (8 days x 96 = 768, minus 4 on the clock change day)


,time,Consumption + Storage,Consumption,Pumping,Hydro,Solar,Import,Export,Wind,Natural Gas,Other Thermal,Biomass,Coal,Wave,Consumption of Batteries,Battery Injection,date
0,00:00,5736.3,5632.6,103.7,3730.4,0.1,0.0,935.3,2296.5,310.3,8.1,341.9,0.0,0.0,0.0,0.0,2025-03-24
1,00:15,5718.2,5585.3,132.9,3493.7,0.1,0.0,749.2,2247.9,391.2,8.1,342.5,0.0,0.0,0.0,0.0,2025-03-24
2,00:30,5676.5,5462.9,213.6,3512.8,0.1,0.0,753.7,2185.4,396.3,8.0,345.1,0.0,0.0,0.0,0.0,2025-03-24
3,00:45,5632.1,5331.9,300.2,3459.3,0.1,0.0,659.8,2114.2,381.3,8.5,345.6,0.0,0.0,0.0,0.0,2025-03-24
4,01:00,6127.9,5230.1,897.8,2926.2,0.1,372.9,0.0,2087.1,401.5,8.4,348.4,0.0,0.0,0.0,0.0,2025-03-24
